# Jedna partycja per device - Zła definicja composite key
Wszystkie dane urządzenia trafiają do jednej partycji, co sprawia, że odczyt staje się coraz wolniejszy.
Dane posortowane są w kolejności od najnowszych do najstarszych.
Partition key = device_id
Clustering key = event_time DESC

In [58]:
import uuid, time
from cassandra.cluster import Cluster

In [59]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

Połączono z Cassandra


In [60]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

Utworzono keyspace


In [61]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_5 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY (device_id, day, event_time) -- Tu mamy błąd w definicji klucza
    ) WITH CLUSTERING ORDER BY (day DESC, event_time DESC)
""")
print("Utworzono tabelę 'users'")

Utworzono tabelę 'users'


## Generate test data

In [62]:
!python ../_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_5 \
  --records 1000 \
  --batch-size 100

Done. Loader=cassandra, Records=1000


In [ ]:
start = time.perf_counter()
rows = session.execute("""
SELECT *
FROM
    events_by_device_5
WHERE
    device_id='device_1'
    AND day='2026-03-29'
LIMIT 10
""")
end = time.perf_counter()
print(f"Zapytanie wykonane w {end - start:.4f} sekund")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

Zapytanie wykonane w 0.0038 sekund
Znalezione rekordy:
- device_1 2026-03-28 (20.90999984741211°C, 57.779998779296875%) - 2026-03-28 16:23:12.011000
- device_1 2026-03-28 (30.440000534057617°C, 40.5099983215332%) - 2026-03-28 16:18:35.846000
- device_1 2026-03-28 (31.969999313354492°C, 60.5%) - 2026-03-28 16:17:53.091000
- device_1 2026-03-28 (31.1200008392334°C, 58.15999984741211%) - 2026-03-28 16:16:33.115000
- device_1 2026-03-28 (31.3700008392334°C, 65.23999786376953%) - 2026-03-28 16:13:30.139000
- device_1 2026-03-28 (34.220001220703125°C, 66.87999725341797%) - 2026-03-28 16:12:33.117000
- device_1 2026-03-28 (34.959999084472656°C, 52.95000076293945%) - 2026-03-28 16:12:30.138000
- device_1 2026-03-28 (25.459999084472656°C, 55.84000015258789%) - 2026-03-28 16:11:30.136000
- device_1 2026-03-28 (33.900001525878906°C, 43.5%) - 2026-03-28 16:10:53.089000
- device_1 2026-03-28 (22.889999389648438°C, 34.099998474121094%) - 2026-03-28 16:10:33.117000


In [64]:
# Zamknięcie połączenia
cluster.shutdown()